# EDA — YouToxic

- **Dropped:** `CommentId`, `VideoId` (not used anywhere in the project)
- **Model input:** `Text`
- **Model target:** `IsToxic` (Essential phase)
- **Kept for analysis / later levels:** all other `Is*` label columns

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.load_data import (
    TARGET_COLUMN,
    label_columns,
    load_dataset,
    prepare_xy,
    without_ids,
)

In [ ]:
raw = load_dataset(ROOT / "data/raw/youtoxic_english_1000.csv")
df = without_ids(raw)
df.head()

In [ ]:
print("Rows:", len(df))
print("Columns:", list(df.columns))
print("Label columns:", label_columns(df))
print("\nMissing Text:", df["Text"].isna().sum())
print("Empty Text:", (df["Text"].astype(str).str.strip() == "").sum())
print("Duplicate texts:", df["Text"].duplicated().sum())

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts()
print(target_counts)
print(f"{TARGET_COLUMN} positive rate: {target_counts.get(True, 0) / len(df):.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
target_counts.plot(kind="bar", ax=ax, color=["#4c78a8", "#e45756"])
ax.set_title(f"{TARGET_COLUMN} distribution")
ax.set_xlabel(TARGET_COLUMN)
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
label_prev = {}
for col in label_columns(df):
    label_prev[col] = df[col].astype(bool).mean()

prev = pd.Series(label_prev).sort_values(ascending=False)
print(prev.apply(lambda x: f"{x:.1%}"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
prev.plot(kind="barh", ax=ax, color="#72b7b2")
ax.set_title("Label prevalence (all Is* columns)")
ax.set_xlabel("Share of comments")
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df[TARGET_COLUMN], df["IsAbusive"], margins=True)

In [ ]:
bool_labels = df[label_columns(df)].astype(bool)
co_occurrence = bool_labels.T.dot(bool_labels)
co_occurrence

In [ ]:
df["text_length"] = df["Text"].astype(str).str.len()
df.groupby(TARGET_COLUMN)["text_length"].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, subset in df.groupby(TARGET_COLUMN):
    ax.hist(subset["text_length"], bins=40, alpha=0.6, label=str(label))
ax.set_title(f"Comment length by {TARGET_COLUMN}")
ax.set_xlabel("Characters")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
x, y = prepare_xy(raw)
print(f"Training samples after prepare_xy: {len(x)}")
print(f"Dropped from raw: {len(df) - len(x)}")
print(f"{TARGET_COLUMN} positive rate: {y.mean():.2%}")

In [ ]:
for label, title in [(1, "Toxic"), (0, "Non-toxic")]:
    print(f"\n=== {title} (first 3) ===")
    for text in x[y == label].head(3):
        print(text[:280], "\n---")

## Takeaways

- **Essential model:** `Text` → `IsToxic` only; auxiliary `Is*` columns inform EDA and future levels (multi-label, error analysis).
- `IsAbusive` and other subtypes overlap with `IsToxic` — useful context, not training features (avoids label leakage).
- Slight class imbalance → `class_weight='balanced'` in training.
- Long comments / URLs → regex preprocessing matters.

# External datasets — Davidson & Jigsaw

The EDA above covers **YouToxic**, the project's core dataset. The model also trains on two extra public hate-speech datasets to learn from more examples. This section documents exactly what is done with them: which columns are kept, how the target is built, and which rows are used.

**Golden rule:** external data feeds the **training set only**. The test set stays 100% YouToxic, so every reported score reflects the real domain (YouTube comments).

| Dataset | Domain | Role |
|---|---|---|
| YouToxic | YouTube comments | train + test |
| Davidson | Twitter | train only |
| Jigsaw | Wikipedia comments | train only |

In [ ]:
import yaml
from src.data.external import (
    load_davidson,
    load_jigsaw,
    load_external_training_data,
    sample_balanced,
)

with open(ROOT / "configs/default.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

ext_cfg = config["external_data"]
ext_cfg

## Davidson

The Davidson dataset is ~25k labelled tweets.

Source columns: an unnamed index, `count`, `hate_speech`, `offensive_language`, `neither`, `class`, `tweet`.

- **Kept:** `tweet` → `Text`
- **Dropped:** the index and the annotator vote counts (`count`, `hate_speech`, `offensive_language`, `neither`); not needed for a binary target.
- **Target:** `class` is `0` hate, `1` offensive, `2` neither. We map **hate + offensive → toxic (1)** and **neither → not toxic (0)**.

In [ ]:
davidson_raw = pd.read_csv(ROOT / ext_cfg["davidson"]["path"])
print("Davidson raw shape:", davidson_raw.shape)
print("Columns:", list(davidson_raw.columns))
davidson_raw["class"].value_counts().sort_index()

In [ ]:
davidson = load_davidson(ROOT / ext_cfg["davidson"]["path"])
print("After mapping to Text / IsToxic:", davidson.shape)
print("Toxic rate:", f"{davidson['IsToxic'].mean():.1%}")
davidson.head()

## Jigsaw

The Jigsaw dataset is ~160k labelled Wikipedia comments.

Source columns: `id`, `comment_text`, and six toxicity tags (`toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate`).

- **Kept:** `comment_text` → `Text`
- **Dropped:** `id`
- **Target:** a comment is **toxic (1) if any of the six tags is set**, otherwise **not toxic (0)**.

In [ ]:
jigsaw_raw = pd.read_csv(ROOT / ext_cfg["jigsaw"]["path"])
print("Jigsaw raw shape:", jigsaw_raw.shape)
print("Columns:", list(jigsaw_raw.columns))

jigsaw = load_jigsaw(ROOT / ext_cfg["jigsaw"]["path"])
print("\nAfter mapping to Text / IsToxic:", jigsaw.shape)
print("Toxic rate:", f"{jigsaw['IsToxic'].mean():.1%}")
jigsaw.head()

## Balanced sampling — which rows we use

The three datasets differ a lot in size and class balance, so we do not use them raw. From each external dataset we take a **balanced sample** (half toxic, half not) of a fixed size set in `configs/default.yaml`. The remaining rows are **not used**.

Why we drop rows on purpose:

- **Balance:** keeps the merged training set close to 50/50.
- **Do not drown YouToxic:** 159k raw Jigsaw rows would bury YouToxic's ~1k.
- **Training time:** ~38k rows train fast, with little loss versus using everything.

In [ ]:
summary = []
for name, loader in [("davidson", load_davidson), ("jigsaw", load_jigsaw)]:
    full = loader(ROOT / ext_cfg[name]["path"])
    taken = sample_balanced(full, ext_cfg[name]["sample_size"], ext_cfg["random_state"])
    summary.append({
        "dataset": name,
        "available": len(full),
        "taken": len(taken),
        "discarded": len(full) - len(taken),
        "toxic_rate_taken": f"{taken['IsToxic'].mean():.0%}",
    })
pd.DataFrame(summary)

In [ ]:
youtoxic_y = prepare_xy(raw)[1]
balance = pd.DataFrame({
    "YouToxic": [1 - youtoxic_y.mean(), youtoxic_y.mean()],
    "Davidson": [1 - davidson["IsToxic"].mean(), davidson["IsToxic"].mean()],
    "Jigsaw": [1 - jigsaw["IsToxic"].mean(), jigsaw["IsToxic"].mean()],
}, index=["not toxic", "toxic"])

fig, ax = plt.subplots(figsize=(7, 4))
balance.T.plot(kind="bar", stacked=True, ax=ax, color=["#4c78a8", "#e45756"])
ax.set_title("Class balance per dataset (full, before sampling)")
ax.set_ylabel("Share")
ax.legend(title="", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [ ]:
lengths = {
    "YouToxic": raw["Text"].astype(str).str.len(),
    "Davidson": davidson["Text"].str.len(),
    "Jigsaw": jigsaw["Text"].str.len(),
}
fig, ax = plt.subplots(figsize=(7, 4))
for name, series in lengths.items():
    ax.hist(series.clip(upper=1000), bins=40, alpha=0.5, label=name, density=True)
ax.set_title("Comment length by dataset (clipped at 1000 chars)")
ax.set_xlabel("Characters")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
combined = load_external_training_data(config)
print("Combined external training data:", len(combined), "rows")
print("Toxic rate:", f"{combined['IsToxic'].mean():.0%}")
print()
print("This is concatenated with the YouToxic *training* split only.")
print("The YouToxic test split never sees external data.")

## Takeaways — external datasets

- The model trains on **3 datasets** but is evaluated on **1** (the YouToxic test split).
- Davidson and Jigsaw are mapped to the same `Text` / `IsToxic` schema as YouToxic.
- From each external dataset we take a **balanced sample**; the rest is dropped on purpose (balance, avoid drowning YouToxic, training time).
- The domains differ (Twitter / Wikipedia vs YouTube), which is exactly why the **test set stays pure YouToxic**.